# 02 — Export CSV-Uploads für Brandwatch

Liest [`data/accounts.csv`](../data/accounts.csv) (die Single-Source-of-Truth
aus Notebook 01) und exportiert CSV-Dateien im Brandwatch-Content-Source-Upload-Format:
- **Eine Spalte** (Handle), **kein Header**, eine Zeile pro Account.

Output nach `output/csv_uploads/`:
- `facebook_alle.csv` — alle Facebook-Handles (eine Datei)
- `instagram_alle.csv` — alle Instagram-Handles (eine Datei)
- `facebook_handles_1.csv ... _N.csv` — Facebook in 100er-Blöcken
- `instagram_handles_1.csv ... _N.csv` — Instagram in 100er-Blöcken
- `stiftungen_facebook.csv`, `stiftungen_instagram.csv` — nur pol. Stiftungen

In [1]:
import os
import math

import pandas as pd

if os.path.basename(os.getcwd()) == "scripts":
    PROJECT_ROOT = os.path.dirname(os.getcwd())
else:
    PROJECT_ROOT = os.getcwd()

DATA_DIR = os.path.join(PROJECT_ROOT, "data")
CSV_DIR = os.path.join(PROJECT_ROOT, "output", "csv_uploads")

ACCOUNTS_CSV = os.path.join(DATA_DIR, "accounts.csv")
STIFTUNGEN_CSV = os.path.join(DATA_DIR, "stiftungen.csv")

os.makedirs(CSV_DIR, exist_ok=True)


def write_handles(handles, filepath):
    """Schreibt Handles als einspaltiges CSV ohne Header."""
    with open(filepath, "w", encoding="utf-8") as f:
        for h in handles:
            f.write(h + "\n")


def export_chunked(handles, prefix, chunk_size=100):
    """Handles in nummerierten 100er-Blöcken schreiben."""
    total = math.ceil(len(handles) / chunk_size)
    for i in range(total):
        chunk = handles[i * chunk_size : (i + 1) * chunk_size]
        path = os.path.join(CSV_DIR, f"{prefix}_{i + 1}.csv")
        write_handles(chunk, path)
        print(f"    {prefix}_{i + 1}.csv ({len(chunk)} Handles)")
    return total

## 1. accounts.csv einlesen

In [2]:
accounts = pd.read_csv(ACCOUNTS_CSV)
print(f"accounts.csv: {len(accounts)} Zeilen")
print(accounts["channel"].value_counts())

accounts.csv: 15828 Zeilen
channel
instagram    4837
x            4742
facebook     4567
tiktok       1500
youtube       182
Name: count, dtype: int64


## 2. Alle Facebook-Handles

In [3]:
fb = (
    accounts.loc[accounts["channel"] == "facebook", "handle"]
    .dropna()
    .drop_duplicates()
    .sort_values(key=lambda s: s.str.lower())
    .tolist()
)
write_handles(fb, os.path.join(CSV_DIR, "facebook_alle.csv"))
print(f"  facebook_alle.csv: {len(fb)} Accounts")
print("  Facebook (100er-Blöcke):")
fb_files = export_chunked(fb, "facebook_handles")

  facebook_alle.csv: 4447 Accounts
  Facebook (100er-Blöcke):
    facebook_handles_1.csv (100 Handles)
    facebook_handles_2.csv (100 Handles)
    facebook_handles_3.csv (100 Handles)
    facebook_handles_4.csv (100 Handles)
    facebook_handles_5.csv (100 Handles)
    facebook_handles_6.csv (100 Handles)
    facebook_handles_7.csv (100 Handles)
    facebook_handles_8.csv (100 Handles)
    facebook_handles_9.csv (100 Handles)
    facebook_handles_10.csv (100 Handles)
    facebook_handles_11.csv (100 Handles)
    facebook_handles_12.csv (100 Handles)
    facebook_handles_13.csv (100 Handles)
    facebook_handles_14.csv (100 Handles)
    facebook_handles_15.csv (100 Handles)
    facebook_handles_16.csv (100 Handles)
    facebook_handles_17.csv (100 Handles)
    facebook_handles_18.csv (100 Handles)
    facebook_handles_19.csv (100 Handles)
    facebook_handles_20.csv (100 Handles)
    facebook_handles_21.csv (100 Handles)
    facebook_handles_22.csv (100 Handles)
    facebook_handles_23

## 3. Alle Instagram-Handles

In [4]:
ig = (
    accounts.loc[accounts["channel"] == "instagram", "handle"]
    .dropna()
    .drop_duplicates()
    .sort_values(key=lambda s: s.str.lower())
    .tolist()
)
write_handles(ig, os.path.join(CSV_DIR, "instagram_alle.csv"))
print(f"  instagram_alle.csv: {len(ig)} Accounts")
print("  Instagram (100er-Blöcke):")
ig_files = export_chunked(ig, "instagram_handles")

print(f"\nGesamt: {fb_files} Facebook-Dateien, {ig_files} Instagram-Dateien")

  instagram_alle.csv: 4587 Accounts
  Instagram (100er-Blöcke):
    instagram_handles_1.csv (100 Handles)
    instagram_handles_2.csv (100 Handles)
    instagram_handles_3.csv (100 Handles)
    instagram_handles_4.csv (100 Handles)
    instagram_handles_5.csv (100 Handles)
    instagram_handles_6.csv (100 Handles)
    instagram_handles_7.csv (100 Handles)
    instagram_handles_8.csv (100 Handles)
    instagram_handles_9.csv (100 Handles)
    instagram_handles_10.csv (100 Handles)
    instagram_handles_11.csv (100 Handles)
    instagram_handles_12.csv (100 Handles)
    instagram_handles_13.csv (100 Handles)
    instagram_handles_14.csv (100 Handles)
    instagram_handles_15.csv (100 Handles)
    instagram_handles_16.csv (100 Handles)
    instagram_handles_17.csv (100 Handles)
    instagram_handles_18.csv (100 Handles)
    instagram_handles_19.csv (100 Handles)
    instagram_handles_20.csv (100 Handles)
    instagram_handles_21.csv (100 Handles)
    instagram_handles_22.csv (100 Handles)

## 4. Stiftungen separat
Selektiert aus `accounts.csv` die Zeilen, deren `name` exakt in der
`Stiftung`-Spalte von `stiftungen.csv` steht. So bleibt `accounts.csv` die
einzige Quelle für Handles — `stiftungen.csv` dient nur als Selektor.

In [5]:
stift = pd.read_csv(STIFTUNGEN_CSV)
stift_names = set(stift["Stiftung"].dropna().astype(str).str.strip())
print(f"Stiftungen (Namen): {len(stift_names)}")

sel = accounts[accounts["name"].isin(stift_names)]
print(f"accounts.csv-Zeilen für Stiftungen: {len(sel)}")

stift_fb = (
    sel.loc[sel["channel"] == "facebook", "handle"]
    .dropna().drop_duplicates()
    .sort_values(key=lambda s: s.str.lower()).tolist()
)
write_handles(stift_fb, os.path.join(CSV_DIR, "stiftungen_facebook.csv"))
print(f"  stiftungen_facebook.csv: {len(stift_fb)} Accounts")

stift_ig = (
    sel.loc[sel["channel"] == "instagram", "handle"]
    .dropna().drop_duplicates()
    .sort_values(key=lambda s: s.str.lower()).tolist()
)
write_handles(stift_ig, os.path.join(CSV_DIR, "stiftungen_instagram.csv"))
print(f"  stiftungen_instagram.csv: {len(stift_ig)} Accounts")

Stiftungen (Namen): 13
accounts.csv-Zeilen für Stiftungen: 36
  stiftungen_facebook.csv: 12 Accounts
  stiftungen_instagram.csv: 12 Accounts


## 5. Zusammenfassung

In [6]:
print(f"Fertig. Alle CSVs liegen in: {CSV_DIR}")
for name in sorted(os.listdir(CSV_DIR)):
    path = os.path.join(CSV_DIR, name)
    if os.path.isfile(path):
        n = sum(1 for _ in open(path, "r", encoding="utf-8"))
        print(f"  {name}  ({n} Zeilen)")

Fertig. Alle CSVs liegen in: /Users/zorbeyozcan/Projekte/query_printer/output/csv_uploads
  facebook_alle.csv  (4447 Zeilen)
  facebook_handles_1.csv  (100 Zeilen)
  facebook_handles_10.csv  (100 Zeilen)
  facebook_handles_11.csv  (100 Zeilen)
  facebook_handles_12.csv  (100 Zeilen)
  facebook_handles_13.csv  (100 Zeilen)
  facebook_handles_14.csv  (100 Zeilen)
  facebook_handles_15.csv  (100 Zeilen)
  facebook_handles_16.csv  (100 Zeilen)
  facebook_handles_17.csv  (100 Zeilen)
  facebook_handles_18.csv  (100 Zeilen)
  facebook_handles_19.csv  (100 Zeilen)
  facebook_handles_2.csv  (100 Zeilen)
  facebook_handles_20.csv  (100 Zeilen)


  facebook_handles_21.csv  (100 Zeilen)
  facebook_handles_22.csv  (100 Zeilen)
  facebook_handles_23.csv  (100 Zeilen)


  facebook_handles_24.csv  (100 Zeilen)
  facebook_handles_25.csv  (100 Zeilen)
  facebook_handles_26.csv  (100 Zeilen)
  facebook_handles_27.csv  (100 Zeilen)
  facebook_handles_28.csv  (100 Zeilen)


  facebook_handles_29.csv  (100 Zeilen)
  facebook_handles_3.csv  (100 Zeilen)
  facebook_handles_30.csv  (100 Zeilen)
  facebook_handles_31.csv  (100 Zeilen)
  facebook_handles_32.csv  (100 Zeilen)


  facebook_handles_33.csv  (100 Zeilen)
  facebook_handles_34.csv  (100 Zeilen)
  facebook_handles_35.csv  (100 Zeilen)
  facebook_handles_36.csv  (100 Zeilen)
  facebook_handles_37.csv  (100 Zeilen)
  facebook_handles_38.csv  (100 Zeilen)
  facebook_handles_39.csv  (100 Zeilen)
  facebook_handles_4.csv  (100 Zeilen)
  facebook_handles_40.csv  (100 Zeilen)
  facebook_handles_41.csv  (100 Zeilen)
  facebook_handles_42.csv  (100 Zeilen)
  facebook_handles_43.csv  (100 Zeilen)
  facebook_handles_44.csv  (100 Zeilen)
  facebook_handles_45.csv  (47 Zeilen)
  facebook_handles_5.csv  (100 Zeilen)
  facebook_handles_6.csv  (100 Zeilen)
  facebook_handles_7.csv  (100 Zeilen)
  facebook_handles_8.csv  (100 Zeilen)


...

  instagram_handles_1.csv  (100 Zeilen)
  instagram_handles_10.csv  (100 Zeilen)
  instagram_handles_11.csv  (100 Zeilen)
  instagram_handles_12.csv  (100 Zeilen)
  instagram_handles_13.csv  (100 Zeilen)
  instagram_handles_14.csv  (100 Zeilen)
  instagram_handles_15.csv  (100 Zeilen)
  instagram_handles_16.csv  (100 Zeilen)
  instagram_handles_17.csv  (100 Zeilen)


  instagram_handles_18.csv  (100 Zeilen)
  instagram_handles_19.csv  (100 Zeilen)
  instagram_handles_2.csv  (100 Zeilen)
  instagram_handles_20.csv  (100 Zeilen)
  instagram_handles_21.csv  (100 Zeilen)


  instagram_handles_22.csv  (100 Zeilen)
  instagram_handles_23.csv  (100 Zeilen)
  instagram_handles_24.csv  (100 Zeilen)
  instagram_handles_25.csv  (100 Zeilen)
  instagram_handles_26.csv  (100 Zeilen)
  instagram_handles_27.csv  (100 Zeilen)
  instagram_handles_28.csv  (100 Zeilen)
  instagram_handles_29.csv  (100 Zeilen)
  instagram_handles_3.csv  (100 Zeilen)
  instagram_handles_30.csv  (100 Zeilen)
  instagram_handles_31.csv  (100 Zeilen)
  instagram_handles_32.csv  (100 Zeilen)
  instagram_handles_33.csv  (100 Zeilen)
  instagram_handles_34.csv  (100 Zeilen)
  instagram_handles_35.csv  (100 Zeilen)
  instagram_handles_36.csv  (100 Zeilen)
  instagram_handles_37.csv  (100 Zeilen)
  instagram_handles_38.csv  (100 Zeilen)
  instagram_handles_39.csv  (100 Zeilen)
  instagram_handles_4.csv  (100 Zeilen)
  instagram_handles_40.csv  (100 Zeilen)
  instagram_handles_41.csv  (100 Zeilen)
  instagram_handles_42.csv  (100 Zeilen)


  instagram_handles_43.csv  (100 Zeilen)
  instagram_handles_44.csv  (100 Zeilen)
  instagram_handles_45.csv  (100 Zeilen)


  instagram_handles_46.csv  (87 Zeilen)
  instagram_handles_5.csv  (100 Zeilen)
  instagram_handles_6.csv  (100 Zeilen)
  instagram_handles_7.csv  (100 Zeilen)
  instagram_handles_8.csv  (100 Zeilen)
  instagram_handles_9.csv  (100 Zeilen)
  stiftungen_facebook.csv  (12 Zeilen)
  stiftungen_instagram.csv  (12 Zeilen)
